# Build Render Montage

Build a montage from a saved rendered-sample directory. The default input is `logs/render_confirm_1000`, and the default image is `rgb.png`.

In [ ]:
from __future__ import annotations

import json
import random
from dataclasses import dataclass
from pathlib import Path

from PIL import Image

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        return None

Image.MAX_IMAGE_PIXELS = None

In [ ]:
INPUT_ROOT = Path("logs/render_confirm_1000")
OUTPUT_PATH = INPUT_ROOT / "montage_rgb_uniform_15x10.png"

IMAGE_NAME = "rgb.png"
ROWS = 10
COLS = 15
TILE_WIDTH = 160
TILE_HEIGHT = 120
GAP_X = 0
GAP_Y = 10
OUTER_PAD = 8
BACKGROUND = 235

# head | tail | uniform | random
SAMPLE_MODE = "uniform"
SORT_BY = "sample_id"
SEED = 0

SAVE_INDEX = True
INDEX_PATH = OUTPUT_PATH.with_suffix(".json")

In [ ]:
@dataclass(frozen=True)
class SampleEntry:
    sample_dir: Path
    image_path: Path
    sample_id: str
    asset_path: str


def load_meta(sample_dir: Path) -> dict:
    meta_path = sample_dir / "meta.json"
    if not meta_path.exists():
        return {}
    return json.loads(meta_path.read_text(encoding="utf-8"))


def collect_samples(input_root: Path, image_name: str, sort_by: str) -> list[SampleEntry]:
    entries = []
    for sample_dir in sorted(path for path in input_root.iterdir() if path.is_dir()):
        image_path = sample_dir / image_name
        if not image_path.exists():
            continue
        meta = load_meta(sample_dir)
        entries.append(
            SampleEntry(
                sample_dir=sample_dir,
                image_path=image_path,
                sample_id=sample_dir.name,
                asset_path=str(meta.get("asset_path", "")),
            )
        )
    if sort_by == "asset_path":
        entries.sort(key=lambda item: (item.asset_path, item.sample_id))
    else:
        entries.sort(key=lambda item: item.sample_id)
    return entries


def pick_samples(entries: list[SampleEntry], limit: int, sample_mode: str, seed: int) -> list[SampleEntry]:
    if len(entries) <= limit:
        return entries
    if sample_mode == "head":
        return entries[:limit]
    if sample_mode == "tail":
        return entries[-limit:]
    if sample_mode == "random":
        rng = random.Random(seed)
        picked = rng.sample(entries, k=limit)
        picked.sort(key=lambda item: item.sample_id)
        return picked
    if limit == 1:
        return [entries[0]]

    max_index = len(entries) - 1
    indices = [round(i * max_index / (limit - 1)) for i in range(limit)]
    return [entries[idx] for idx in indices]


def build_canvas(
    samples: list[SampleEntry],
    rows: int,
    cols: int,
    tile_width: int,
    tile_height: int,
    gap_x: int,
    gap_y: int,
    outer_pad: int,
    background: int,
) -> Image.Image:
    canvas_width = outer_pad * 2 + cols * tile_width + max(cols - 1, 0) * gap_x
    canvas_height = outer_pad * 2 + rows * tile_height + max(rows - 1, 0) * gap_y
    canvas = Image.new("RGB", (canvas_width, canvas_height), color=(background, background, background))

    for idx, sample in enumerate(samples[: rows * cols]):
        row = idx // cols
        col = idx % cols
        x = outer_pad + col * (tile_width + gap_x)
        y = outer_pad + row * (tile_height + gap_y)
        tile = Image.open(sample.image_path).convert("RGB")
        tile = tile.resize((tile_width, tile_height), Image.Resampling.LANCZOS)
        canvas.paste(tile, (x, y))
    return canvas


In [ ]:
assert INPUT_ROOT.exists(), f"Input root not found: {INPUT_ROOT}"
assert ROWS > 0 and COLS > 0
assert TILE_WIDTH > 0 and TILE_HEIGHT > 0
assert 0 <= BACKGROUND <= 255

entries = collect_samples(INPUT_ROOT, IMAGE_NAME, SORT_BY)
limit = ROWS * COLS
picked = pick_samples(entries, limit=limit, sample_mode=SAMPLE_MODE, seed=SEED)

print(f"found samples: {len(entries)}")
print(f"selected samples: {len(picked)}")
print(f"output: {OUTPUT_PATH}")
picked[:5]

In [ ]:
canvas = build_canvas(
    picked,
    rows=ROWS,
    cols=COLS,
    tile_width=TILE_WIDTH,
    tile_height=TILE_HEIGHT,
    gap_x=GAP_X,
    gap_y=GAP_Y,
    outer_pad=OUTER_PAD,
    background=BACKGROUND,
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
canvas.save(OUTPUT_PATH)
display(canvas)

if SAVE_INDEX:
    payload = {
        "input_root": str(INPUT_ROOT),
        "output": str(OUTPUT_PATH),
        "image_name": IMAGE_NAME,
        "rows": ROWS,
        "cols": COLS,
        "tile_width": TILE_WIDTH,
        "tile_height": TILE_HEIGHT,
        "gap_x": GAP_X,
        "gap_y": GAP_Y,
        "outer_pad": OUTER_PAD,
        "background": BACKGROUND,
        "sample_mode": SAMPLE_MODE,
        "sort_by": SORT_BY,
        "samples": [
            {
                "sample_id": item.sample_id,
                "sample_dir": str(item.sample_dir),
                "asset_path": item.asset_path,
            }
            for item in picked
        ],
    }
    INDEX_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"saved index: {INDEX_PATH}")

## Notes

- Change `IMAGE_NAME` to switch to `rgb_masked.png`, `side_overlay.png`, or `nocs_visualization.png`.
- Change `ROWS`, `COLS`, `TILE_WIDTH`, and `TILE_HEIGHT` to match the target layout.
- `SAMPLE_MODE="uniform"` samples evenly from a large batch. Use `head` to keep the first samples.